In [ ]:
import pandas as pd
from pathlib import Path
import os
import plotly.express as px
import geopandas as geopd
from data_preparation.utils.utils import get_s3, get_mapping

In [ ]:
PATH_DATA = Path(os.getcwd()).parent / 'Output' / "data"
PATH_RAW_DATA = PATH_DATA / "raw_data"
PATH_PHEN = PATH_DATA / "final_data"

assert PATH_RAW_DATA.exists(), PATH_RAW_DATA
vodafone_file = PATH_RAW_DATA / "vodafone_attendences.csv"

PATH_PROCESSED = PATH_DATA / "data_processed"
assert PATH_PROCESSED.exists(), PATH_PROCESSED
phen_presenze = PATH_PHEN / "phen_presenze.parquet"

PATH_MAPPING = PATH_RAW_DATA / "mapping"
presenze_trentino_ispat = PATH_RAW_DATA / "presenze_Trentino_ISPAT.csv"
presenze_trentino_ispat_xalb = PATH_RAW_DATA / "presenze_Trentino_ISPAT_alb_xalb.csv"

In [ ]:
def atp_to_comuni():
    id_to_apt = {}
    for apt, id_list in json_apt.items():
        for id_comune in id_list:
            # Trasformiamo in int e poi in stringa a 6 cifre: "22001" -> "022001"
            id_6_cifre = str(int(id_comune)).zfill(6)
            id_to_apt[id_6_cifre] = apt
    return id_to_apt

In [ ]:
presenze = pd.read_parquet(phen_presenze)
vodafone_presenze = pd.read_csv(vodafone_file)
geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")
json_apt = get_mapping("map_comuni_into_apt.json")

In [ ]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_presenze["comune"] = vodafone_presenze["locId"].astype(str).map(location_map)
vodafone_presenze["ID_COMUNE"] = vodafone_presenze["comune"].map(json_vodafone)

# Vigo di Fassa e Pozza di Fassa -> San Giovanni di Fassa
mask = vodafone_presenze["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_presenze.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA"
vodafone_presenze.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_presenze.index[mask]
)

vodafone_presenze_comuni = vodafone_presenze[
    vodafone_presenze["locType"] == "TN_MKT_AL_3"
]

In [ ]:
# comuni Vodafone senza ID
print("locId senza ID_COMUNE:",
      vodafone_presenze_comuni[vodafone_presenze_comuni['ID_COMUNE'].isna()].locId.unique())
vodafone_presenze_comuni = vodafone_presenze_comuni.dropna(subset=['ID_COMUNE'])

In [ ]:
df_aggregato = (
    vodafone_presenze_comuni
    .groupby(['date', 'comune', 'userProfile'])
    .agg({'value': 'sum', 'ID_COMUNE': 'first'})
    .reset_index()
)

In [ ]:
# correlations 
commuter = df_aggregato[df_aggregato['userProfile'] == 'COMMUTER']
corr_commuter = commuter.pivot(index='date', columns='comune', values='value').corr()

tourist = df_aggregato[df_aggregato['userProfile'] == 'TOURIST'].copy()
corr_tourist = tourist.pivot(index='date', columns='comune', values='value').corr()

In [ ]:
# presenze_alb (pipeline) vs ISPAT per APT
presenze['DATA'] = pd.to_datetime(presenze['DATA'])
presenze['MESE'] = presenze['DATA'].dt.to_period("M")
df_grouped = presenze.groupby(["MESE", "ID_COMUNE"])['presenze_alb'].sum().reset_index()

df_grouped['ID_COMUNE'] = df_grouped['ID_COMUNE'].astype(str).str.zfill(6)
df_grouped['APT'] = df_grouped['ID_COMUNE'].map(atp_to_comuni())
print("Comuni senza APT:", df_grouped.loc[df_grouped['APT'].isna(), 'ID_COMUNE'].unique())
df_grouped.to_csv("grouped.csv", index=False)

In [ ]:
df_grouped['Anno'] = df_grouped['MESE'].dt.year
df_grouped['Mese'] = df_grouped['MESE'].dt.month
df_grouped_pres_summed = (
    df_grouped.groupby(["APT", "Anno", "Mese"])['presenze_alb'].sum()
    .reset_index().rename(columns={'APT': 'Ambito'})
)

pres_off_df_apt = pd.read_csv(presenze_trentino_ispat).sort_values("Ambito")
mg = pres_off_df_apt.merge(df_grouped_pres_summed, on=['Anno', 'Mese', 'Ambito'])
print(f"Righe ISPAT: {len(pres_off_df_apt)} | righe con corrispondenza: {len(mg)}")
mg['diff'] = mg['Presenze'] - mg['presenze_alb']
print(mg['diff'].describe())
mg[abs(mg['diff']) > 15]


In [ ]:
pres_off_df_apt_noprov = pres_off_df_apt[pres_off_df_apt['Ambito'] != 'Provincia'].copy()
summed = pres_off_df_apt_noprov.groupby(["Anno", "Mese"])['Presenze'].sum().reset_index()

pres_off_xalb = pd.read_csv(presenze_trentino_ispat_xalb)
mgd = pres_off_xalb.merge(summed, on=['Anno', 'Mese'])
mgd['diff'] = mgd['Presenze alberghi'] - mgd['Presenze']
print(mgd['diff'].describe())


In [ ]:
# split tourists equally
tourist['date'] = pd.to_datetime(tourist['date'])
tourist['mese'] = tourist['date'].dt.to_period('M')

tourist_ids = tourist[['date', 'mese', 'comune', 'ID_COMUNE', 'value']].copy()
tourist_ids['num_ids'] = tourist_ids['ID_COMUNE'].apply(lambda x: len(x) if isinstance(x, list) else 1)
tourist_ids['value'] = tourist_ids['value'] / tourist_ids['num_ids']

tourist_ids = tourist_ids.explode('ID_COMUNE').dropna(subset=['ID_COMUNE'])
tourist_ids['ID_NORM'] = tourist_ids['ID_COMUNE'].astype(str).str.strip().str.lstrip('0')

tourist_month = (
    tourist_ids.groupby(['mese', 'ID_NORM'], as_index=False)['value']
    .sum()
    .rename(columns={'value': 'presenze_vodafone'})
)

In [ ]:
#  ISPAT per APT
pres_off_df_apt_noprov['mese'] = pd.to_datetime(
    pres_off_df_apt_noprov['Anno'].astype(str) + '-' + pres_off_df_apt_noprov['Mese'].astype(str).str.zfill(2)
).dt.to_period('M')
pres_off_df_apt_noprov['ID'] = pres_off_df_apt_noprov['Ambito'].map(json_apt)
print("Ambiti senza lista comuni:", pres_off_df_apt_noprov.loc[pres_off_df_apt_noprov['ID'].isna(), 'Ambito'].unique())

df_esploso_apt_no_prov = pres_off_df_apt_noprov[['Ambito', 'mese', 'ID']].explode('ID')
df_esploso_apt_no_prov = df_esploso_apt_no_prov.dropna(subset=['ID']).copy()
df_esploso_apt_no_prov['ID_NORM'] = df_esploso_apt_no_prov['ID'].astype(str).str.strip().str.lstrip('0')

# checks 
print("Comuni degli ambiti senza Vodafone:",
      sorted(set(df_esploso_apt_no_prov['ID_NORM']) - set(tourist_month['ID_NORM'])))
print("Comuni Vodafone senza ambito:",
      sorted(set(tourist_month['ID_NORM']) - set(df_esploso_apt_no_prov['ID_NORM'])))
print("Comuni presenti in piu' ambiti nello stesso mese:",
      df_esploso_apt_no_prov.duplicated(['mese', 'ID_NORM']).sum())


In [ ]:
#  Vodafone per ambito e mese
merged_apt_voda = df_esploso_apt_no_prov.merge(tourist_month, on=['mese', 'ID_NORM'], how='left')
df_grouped_apt = merged_apt_voda.groupby(['Ambito', 'mese'])['presenze_vodafone'].sum().reset_index()
cm = merged_apt_voda[merged_apt_voda['mese'] < '2024-04']
cm[cm.presenze_vodafone.isna()]  # 22078 DRENA, 22191 TENNO, 22018 BOCEGNAGO, 22164 SAGRON-MIS, 22059 CINTE TESINO


In [ ]:
tourist_ids[(tourist_ids['ID_COMUNE'] == 22078) & (tourist_ids['mese'] == '2022-03')]
tourist_ids[(tourist_ids['ID_COMUNE'] == 22059) & (tourist_ids['mese'] == '2022-01')]

In [ ]:
mgdt = pres_off_df_apt_noprov.merge(df_grouped_apt, on=['mese', 'Ambito'], how='outer', indicator=True)
print(mgdt['_merge'].value_counts())
mgdt = mgdt[mgdt['_merge'] == 'both'].drop(columns='_merge')
mgdt = mgdt[mgdt['presenze_vodafone'] > 0].copy()  # evita divisioni per zero / mesi senza dato Vodafone
mgdt['ratio'] = mgdt['Presenze'] / mgdt['presenze_vodafone']


In [ ]:
PROV = (
    mgdt.groupby('mese')
    .agg(presenze_ispat=('Presenze', 'sum'), presenze_vodafone=('presenze_vodafone', 'sum'))
    .reset_index()
)
PROV['ratio'] = PROV['presenze_ispat'] / PROV['presenze_vodafone']
APT = mgdt[['Ambito', 'mese', 'ratio']]

In [ ]:
# APT vs provincia 
confronto = APT.merge(PROV[['mese', 'ratio']], on='mese', suffixes=('_ambito', '_prov'))
confronto['scarto_assoluto'] = confronto['ratio_ambito'] - confronto['ratio_prov']
confronto['scarto_relativo'] = confronto['scarto_assoluto'] / confronto['ratio_prov']
confronto = confronto.sort_values('scarto_relativo', key=abs, ascending=False)
confronto['mese_str'] = confronto['mese'].astype(str)

#  Grafico
fig = px.line(
    confronto.sort_values('mese'),
    x='mese_str', y='ratio_ambito', color='Ambito',
    title='Rapporto Presenze alberghiere ISPAT / Vodafone turisti, per ambito nel tempo'
)
prov_ref = PROV.sort_values('mese')
fig.add_scatter(
    x=prov_ref['mese'].astype(str), y=prov_ref['ratio'],
    mode='lines', name='Rapporto provinciale (somma ambiti)',
    line=dict(color='black', dash='dash')
)
fig.show()